# 23 - Scale confirmation, SEED 1 (robustness of the conv-at-scale surprise)

Seed 0 (notebook 22) gave a surprise: at ~30M params on WikiText-103 BPE, the depthwise-conv
variant BEAT the modern combo by ~9.8% (the opposite of small char scale, where it did not help).
Differential attention stayed consistent (still hurt). This notebook re-runs the same four configs
with **seed 1** to check the conv win is robust, not seed luck. Combined with seed 0, this gives
mean +/- std at ~30M for the paper. Power-cut resilient (checkpoint every 500 iters, atomic; resume).


In [1]:
%matplotlib inline
import os, math, time, csv, json, gc
from contextlib import nullcontext
import numpy as np
import torch
from model import Config, GPT, count_params

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, torch.cuda.get_device_name(0) if device == "cuda" else "")

D = "../dataset"
train_data = torch.from_numpy(np.load(D + "/wt103_bpe_train.npy"))
val_data = torch.from_numpy(np.load(D + "/wt103_bpe_val.npy"))
vocab = json.load(open(D + "/wt103_bpe_meta.json"))["vocab"]
print(f"vocab={vocab}  train_tokens={len(train_data):,}  val_tokens={len(val_data):,}")

BLOCK, BATCH = 256, 16
SEED = 1
CKPT_DIR = "../results/scale_ckpt_s1"
os.makedirs(CKPT_DIR, exist_ok=True)

def get_batch(split, gen):
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - BLOCK, (BATCH,), generator=gen)
    x = torch.stack([d[i:i + BLOCK] for i in ix])
    y = torch.stack([d[i + 1:i + 1 + BLOCK] for i in ix])
    return x.to(device), y.to(device)

def autocast_ctx():
    return torch.autocast(device_type="cuda", dtype=torch.bfloat16) if device == "cuda" else nullcontext()

def train_one(name, overrides, seed=SEED, max_iters=6000, eval_iters=200, lr=3e-4, warmup=200,
              min_lr=3e-5, ckpt_every=500):
    ckpt_path = f"{CKPT_DIR}/{name}.pt"
    torch.manual_seed(seed)
    cfg = Config(vocab_size=vocab, block_size=BLOCK, n_layer=8, n_head=8, n_embd=512,
                 dropout=0.1, **overrides)
    model = GPT(cfg).to(device)
    pm = count_params(model) / 1e6
    opt = torch.optim.AdamW(model.parameters(), lr=lr, betas=(0.9, 0.99), weight_decay=0.1)
    tg = torch.Generator().manual_seed(7777 + seed)
    start_it = 0
    if os.path.exists(ckpt_path):
        ck = torch.load(ckpt_path, map_location=device)
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"])
        tg.set_state(ck["tg"]); start_it = ck["iter"]
        print(f"    [{name}] RESUMED from iter {start_it}/{max_iters}", flush=True)

    def get_lr(it):
        if it < warmup:
            return lr * (it + 1) / warmup
        r = (it - warmup) / max(1, max_iters - warmup)
        return min_lr + 0.5 * (lr - min_lr) * (1 + math.cos(math.pi * r))

    def save_ckpt(it):
        tmp = ckpt_path + ".tmp"
        torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "iter": it,
                    "tg": tg.get_state()}, tmp)
        os.replace(tmp, ckpt_path)

    @torch.no_grad()
    def ev():
        model.eval(); g = torch.Generator().manual_seed(4242); tot = 0.0
        for _ in range(eval_iters):
            x, y = get_batch("val", g)
            with autocast_ctx():
                _, loss = model(x, y)
            tot += loss.item()
        model.train(); return tot / eval_iters

    t0 = time.time()
    for it in range(start_it, max_iters):
        for grp in opt.param_groups:
            grp["lr"] = get_lr(it)
        x, y = get_batch("train", tg)
        with autocast_ctx():
            _, loss = model(x, y)
        opt.zero_grad(set_to_none=True); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        if it % 500 == 0 or it == max_iters - 1:
            print(f"    [{name}] iter {it:5d}/{max_iters}  train_loss {loss.item():.3f}  "
                  f"{(time.time()-t0)/60:.1f} min", flush=True)
        if (it + 1) % ckpt_every == 0:
            save_ckpt(it + 1)
    ppl = math.exp(ev())
    if os.path.exists(ckpt_path):
        os.remove(ckpt_path)
    del model, opt
    gc.collect(); torch.cuda.empty_cache()
    return ppl, pm

print("ready  (SEED=%d, BLOCK=%d BATCH=%d, checkpoint every 500)" % (SEED, BLOCK, BATCH))


device: cuda NVIDIA GeForce RTX 3070
vocab=8192  train_tokens=14,923,628  val_tokens=281,169
ready  (SEED=1, BLOCK=256 BATCH=16, checkpoint every 500)


In [2]:
COMBO = {"pos_encoding": "rope", "activation": "swiglu", "norm_placement": "pre", "norm_type": "rmsnorm"}
CONFIGS = {
    "baseline":    {},
    "combo":       COMBO,
    "combo_conv3": {**COMBO, "conv_kernel": 3},
    "combo_diff":  {**COMBO, "attention": "diff"},
}
MAX_ITERS = 6000

runs_csv = "../results/scale_confirm_s1_runs.csv"
summary = {}
if os.path.exists(runs_csv):
    with open(runs_csv) as f:
        r = csv.reader(f); next(r, None)
        for row in r:
            if len(row) == 3:
                summary[row[0]] = (float(row[2]), float(row[1]))
if not os.path.exists(runs_csv):
    with open(runs_csv, "w", newline="") as f:
        csv.writer(f).writerow(["config", "params_M", "val_ppl"])
if summary:
    print("RESUMING - configs already done:", list(summary), flush=True)

t0 = time.time()
for name, over in CONFIGS.items():
    if name in summary:
        print(f"=== {name} (skip, done: ppl {summary[name][0]:.3f}) ===", flush=True)
        continue
    print(f"=== {name} (seed {SEED}) ===", flush=True)
    p, pm = train_one(name, over, max_iters=MAX_ITERS)
    summary[name] = (p, pm)
    with open(runs_csv, "a", newline="") as f:
        csv.writer(f).writerow([name, f"{pm:.3f}", f"{p:.4f}"])
    print(f"{name:12s} DONE  ppl {p:.3f}  ({pm:.1f}M)  | total {(time.time()-t0)/60:.1f} min\n", flush=True)
print(f"ALL DONE in {(time.time()-t0)/60:.1f} min")


=== baseline (seed 1) ===


    [baseline] iter     0/6000  train_loss 9.104  0.0 min


    [baseline] iter   500/6000  train_loss 5.663  0.6 min


    [baseline] iter  1000/6000  train_loss 5.239  1.2 min


    [baseline] iter  1500/6000  train_loss 5.015  1.8 min


    [baseline] iter  2000/6000  train_loss 4.726  2.4 min


... (51 outputs left out)

    [combo_diff] iter  5500/6000  train_loss 4.054  11.8 min


    [combo_diff] iter  5999/6000  train_loss 4.001  12.9 min


combo_diff   DONE  ppl 56.730  (29.4M)  | total 40.3 min



ALL DONE in 40.3 min


In [3]:
# combine seed 0 + seed 1 -> mean for the robustness check
import numpy as np, csv
def load(fn):
    d = {}
    with open(fn) as f:
        r = csv.reader(f); next(r, None)
        for row in r:
            if len(row) == 3: d[row[0]] = float(row[2])
    return d
s0 = load("../results/scale_confirm_runs.csv")
s1 = load("../results/scale_confirm_s1_runs.csv")
print(f"{'config':12s} {'seed0':>8s} {'seed1':>8s} {'mean':>8s} {'vs combo(mean)':>15s}")
combo_mean = np.mean([s0['combo'], s1['combo']])
for name in ['baseline','combo','combo_conv3','combo_diff']:
    m = np.mean([s0[name], s1[name]])
    print(f"{name:12s} {s0[name]:8.3f} {s1[name]:8.3f} {m:8.3f} {100*(m-combo_mean)/combo_mean:+14.2f}%")
conv_m = np.mean([s0['combo_conv3'], s1['combo_conv3']])
print(f"\nCONV vs COMBO across 2 seeds: {100*(conv_m-combo_mean)/combo_mean:+.2f}%")
print("ROBUST WIN" if (s0['combo_conv3']<s0['combo'] and s1['combo_conv3']<s1['combo']) else "NOT robust (seeds disagree)")


config          seed0    seed1     mean  vs combo(mean)
baseline       61.787   62.023   61.905         +11.63%
combo          55.136   55.774   55.455          +0.00%
combo_conv3    49.722   49.838   49.780         -10.23%
combo_diff     58.548   56.730   57.639          +3.94%

CONV vs COMBO across 2 seeds: -10.23%
ROBUST WIN


## Reading this

If `combo_conv3 < combo` on **both** seeds, the conv-at-scale win is robust and becomes a genuine
positive finding (a depthwise causal conv improves a modern decoder at ~30M on subword data) - which
would re-orient the paper from a pure negative-results study toward a real contribution. If the seeds
disagree, it was seed luck and the small-scale conclusion stands. Either outcome is reported honestly.
